In [ ]:
import re
import json
from bs4 import BeautifulSoup

## Open the json and fill a dictionary

In [ ]:
# def open_monsters():
#     with open('./data/monsters_pretty.json', 'r', encoding="utf-8") as f_:
#         monsters_dict = json.load(f_)
#     return monsters_dict

def open_monsters():
    with open('./data/monsters_pretty.json', 'r', encoding="utf-8") as f_:
        monsters_dict = json.load(f_)
    return {k.replace('\u200e', '').strip(): v for k, v in monsters_dict.items()}

monsters_dict = open_monsters()

In [ ]:

# monster_text = monsters_dict["outsiders_demon_balor"]
# print(monster_text)

print(monsters_dict["undead_bloody-skeletal-champion-advanced-hound-archon"])

## remove top lines up to <h1>

In [ ]:
top_patterns = [r'^.*?(?=<h1)',
                r'<script[^>]*>.*?</script>',
                r'<div class="toc_light_blue[^>]*>.*?</div>'
                ]

def scrub(monsters_dict, scrub_patterns):
    for monster in monsters_dict:
        for pattern in scrub_patterns:
            monsters_dict[monster] = re.sub(pattern, '', monsters_dict[monster], flags=re.DOTALL)

scrub(monsters_dict, top_patterns)



print(monsters_dict["outsiders_demon_balor"])

## Check if it is a category or monster

In [ ]:
def check_has_child(monsters_dict):
    has_child = []
    no_child = []
    for entry in monsters_dict:
        match = re.search(r'<div class="ogn-childpages"', monsters_dict[entry])
        if match:
            has_child.append(entry)
        else:
            no_child.append(entry)
    return has_child, no_child

has_child, no_child = check_has_child(monsters_dict)

print(f"{len(has_child)} {len(no_child)}")

In [ ]:
def check_if_monster(has_child):
    has_child_has_stats = []
    has_child_no_stats = []
    for entry in has_child:
        match = re.search(r'<p class="divider">\s*STATISTICS\s*</p>', monsters_dict[entry])
        # match = re.search(r'<div class="statblock">', monsters_dict[entry])
        if match:
            has_child_has_stats.append(entry)
        else:
            has_child_no_stats.append(entry)
            
    return has_child_has_stats, has_child_no_stats

has_child_has_stats, has_child_no_stats = check_if_monster(has_child)

# print(monsters_dict[has_child_no_stats[1]])
# print(f"{len(has_child_has_stats)} {len(has_child_no_stats)}")
print(f" no child {len(no_child)} \n has child, no stats {len(has_child_has_stats)} \n has child has stats {len(has_child_no_stats)}")


In [ ]:
def split_dict(original_dict, split_list):
    split_dict = {}
    for entry in split_list:
        split_entry = original_dict.pop(entry)
        split_dict.update({entry: split_entry})
    print(f'original: {len(original_dict)}')
    print(f'split: {len(split_dict)}')
    return original_dict, split_dict

monsters_dict, categories_dict = split_dict(monsters_dict, has_child_no_stats)


## check if monster is present twice

In [ ]:
for monster in monsters_dict:
    match = re.search(r'<div class="ogn-childpages" style="float:right">.*?</div>', monsters_dict[monster], flags=re.DOTALL)    
    if match:
        pass
    name = monster.split("_")
    if name[-1] in name[:-1]:
        print(name)

## Remove Childpages Div

In [ ]:
scrub_patterns = [r'<div class="ogn-childpages" style="float:right">.*?</div>', r'<div class="product-right">.*?</div>']

scrub(monsters_dict, scrub_patterns)
print(len(monsters_dict))

## check if statblock

In [ ]:
statblock_pattern = r'<div class="statblock">(.*?)</div>'
thirdpp_pattern = r'3pp'
description_pattern = r'<p class="description">.*?<p>'

def check_content(monsters_dict, pattern):
    yes = []
    no = []
    for monster in monsters_dict:
        monster_text = monsters_dict[monster]
        match = re.search(pattern, monster_text, flags=re.DOTALL | re.IGNORECASE)
        if match:
            yes.append(monster)
        else:
            no.append(monster)
    return yes, no


third_party, vanilla = check_content(monsters_dict, thirdpp_pattern)
print("3pp y/n:", len(third_party), len(vanilla))

yes_statblock, no_statblock = check_content(monsters_dict, statblock_pattern)
print("statblock y/n:", len(yes_statblock), len(no_statblock))

yes_description, no_description = check_content(monsters_dict, description_pattern)
print("description y/n:", len(yes_description), len(no_description))

## try to Identify the Title/Header/Creature block

In [ ]:
title_header_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
h1_pattern = r'<h1>.*?</h1>'

yes, no = check_content(monsters_dict, h1_pattern)
print("h1 y/n:", len(yes), len(no))
yes_head, no_head = check_content(monsters_dict, title_header_pattern)
print("title/header y/n:", len(yes_head), len(no_head))
monsters_dict, monsters_table_dict = split_dict(monsters_dict, no_head)








In [ ]:
for monster in monsters_dict:
    match = re.search(r'<p class="header">\s*APPEARANCE\s*</p>', monsters_dict[monster], re.DOTALL)
    if match:
        print(monster)
        re.sub(r'<p class="header">\s*APPEARANCE\s*</p>', '', monsters_dict[monster], re.DOTALL)

## split off Mythic creatures

In [ ]:
mythic_list = []

for monster in monsters_dict:
    head = re.search(r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>', monsters_dict[monster], re.DOTALL).group(0)
    if re.search(r'Mythic', head) and re.search(r'MR', head):
        mythic_list.append(monster)

monsters_dict, monsters_mythic_dict = split_dict(monsters_dict, mythic_list)

## Summary

In [ ]:
print(len(monsters_dict),len(monsters_mythic_dict),len(monsters_table_dict),len(categories_dict) )

# Parsing

In [ ]:
def clean_text(text):
    clean = re.sub(r'<[^>]+>', '', text).strip()
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

#### save h1 and description

In [ ]:
h1_pattern = r'<h1>(.*?)</h1>'



def extract_title(monsters_dict):
    monsters_data ={}
    for monster in monsters_dict:
        match = re.search(h1_pattern, monsters_dict[monster], re.DOTALL)
        match = match.group(0).replace("\u200e", "")
        title = clean_text(match)
        monsters_data.update({monster: {"title1": title,
                                        "title2": monster.split("_")[-1]}})
        
        monsters_dict[monster] = re.sub(r'<h1>.*?</h1>', '', monsters_dict[monster], flags=re.DOTALL).strip()

    print(len(monsters_data), len(monsters_dict))
    return(monsters_data)


monsters_data = extract_title(monsters_dict)

for monster in monsters_data:
    print(monsters_data[monster]["title1"])

In [ ]:
for monster in monsters_dict:
    match = re.search(r'<p class="description">(.*?)</p>', monsters_dict[monster], flags=re.DOTALL)   
    if match:
        desc = match.group(1).strip()
        desc = clean_text(desc)
        if len(desc) < 1:
            desc = re.search(r'<p class="description">.*?</p>(.*?)</p>', monsters_dict[monster], flags=re.DOTALL).group(1)
            desc = clean_text(desc)
        if len(desc) > 0 and desc[0] == "X":
            monsters_data[monster]["types_"] = desc
            monsters_data[monster]["desc_short"] = ""
        else:
            monsters_data[monster]["desc_short"] = desc
        monsters_dict[monster] = re.sub(r'<p class="description">.*?</p>', '', monsters_dict[monster], flags=re.DOTALL).strip()
    else:
        # monsters_data.update({monster: {"desc_short": ""}})
        monsters_data[monster]["desc_short"] = ""



#### save source and remove trail

In [ ]:
for monster in monsters_dict:
    match = re.search(r'Copyright.*$', monsters_dict[monster], flags=re.DOTALL | re.IGNORECASE)
    if match:
        # print(match.group(0))
        match_href = re.search(r'<a(.*?)href=.*?>(.*?)</a>', match.group(0), flags=re.DOTALL)
        match_p = re.search(r'<p.*?>(.*?)</p>', match.group(0), flags=re.DOTALL)
        match_i = re.search(r'<i.*?>(.*?)</i>', match.group(0), flags=re.DOTALL)
        match_div = re.search(r'<div.*?>(.*?)</div>', match.group(0), flags=re.DOTALL)
        if match_href:
            monsters_data[monster]['source'] = match_href.group(1).strip()
        elif match_p:
            source = re.sub(r'<[^>]+>', '', match_p.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
            # print(source)
        elif match_i:
            source = re.sub(r'<[^>]+>', '', match_i.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
        elif match_div:
            source = re.sub(r'<[^>]+>', '', match_div.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
            monsters_data[monster]['source'] = source
        else:
            monsters_data[monster]['source'] = ""   
    else:
        monsters_data[monster]['source'] = ""


        

In [ ]:
for monster in monsters_dict:
    monsters_dict[monster] = re.sub(
        r'<!--div style="clear:both">.*$',
        '', 
        monsters_dict[monster], 
        flags=re.DOTALL
    )
    # print(repr(monsters_dict[monster][-500:]))
    monsters_dict[monster] = re.sub(
        r'<div class="section15">.*$',
        '', 
        monsters_dict[monster], 
        flags=re.DOTALL
    )
        
monster_text = monsters_dict["humanoids_hobgoblin-commander"]
print(monster_text)



#### Remove APPEARANCE in top_

In [ ]:
def remove_APPEARANCE(monsters_dict):
    top_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
    for monster in monsters_dict:
        match = re.search(top_pattern, monsters_dict[monster], flags=re.DOTALL)
        top_ = clean_text(match.group(0))
        if top_ == "APPEARANCE":
            print(clean_text(monsters_dict[monster][:240]))
            monsters_dict[monster] = monsters_dict[monster].replace(match.group(0), '', 1)
    return monsters_dict

monsters_dict = remove_APPEARANCE(monsters_dict)


In [ ]:
top_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
for monster in monsters_dict:
    match = re.search(top_pattern, monsters_dict[monster], flags=re.DOTALL)
    top_ = clean_text(match.group(0))
    monsters_data[monster]["top_"] = top_
    monsters_dict[monster] = monsters_dict[monster].replace(match.group(0), '', 1)
    

In [ ]:
# for monster in monsters_data:
#     print(monster)
#     print(monsters_data[monster]["title1"])
#     print(monsters_data[monster]["desc_short"])
#     print(monsters_data[monster]["top_"])
#     print("\n")

for monster in monsters_data:
    if monsters_data[monster]["desc_short"] == "":
        print(monster)

print(monsters_dict["undead_bloody-skeletal-champion-advanced-hound-archon"])
# for monster in monsters_dict:
#     print(monsters_dict[monster])

In [ ]:
def clean_text(text):
    clean = re.sub(r'<[^>]+>', '', text).strip()
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

def extract_section(monsters_dict, monsters_data):
    bads = []
    for monster in monsters_dict:
        types_pattern = r'^(.*?)(?=<p class="(?:divider|header)">\s*DEFENSES?\s*</p>)'
        types_pattern_2 = r'^(.*?)(?=<b>\s*DEFENSES?\s*</b>)'
        match = re.search(types_pattern, monsters_dict[monster], flags=re.DOTALL | re.IGNORECASE)
        match2 = re.search(types_pattern_2, monsters_dict[monster], flags=re.DOTALL | re.IGNORECASE)
        if match:
            clean = clean_text(match.group(0))
            if clean:
                monsters_data[monster]["types_"] = clean
            if not clean:
                # print(monster, "\n", monsters_dict[monster])
                bads.append(monster)
        # elif match2:
        #     clean = clean_text(match2.group(0))
            # print(clean)?
        else:
            bads.append(monster)
    return bads
# def extract_types(text, monster):
#     if len(text) == 0:
#         print(monster)


bads = extract_section(monsters_dict, monsters_data)
for bad in bads:
    print(monsters_data[bad])
# print(monsters_dict["humanoids_gnoll"])
# print(monsters_data["humanoids_gnoll"])

In [ ]:
monster_text = monsters_dict["outsiders_demon_balor"]
print(monsters_data["outsiders_demon_balor"])
print(monster_text)